In [2]:
python -c "import torch; print('PyTorch:',torch.__version__); print('CUDA available:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not detected')"

SyntaxError: invalid syntax (2586979009.py, line 1)

In [3]:
python -c "import torch; print('PyTorch:',torch.__version__); print('CUDA available:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not detected')"

SyntaxError: invalid syntax (2586979009.py, line 1)

In [4]:
python -m pip install transformers==4.51.3 trl==0.15.2 peft==0.15.2 accelerate==1.2.1 datasets==3.6.0 dill==0.3.8 multiprocess==0.70.16 bitsandbytes==0.48.1 scikit-learn pandas pyarrow sentencepiece

SyntaxError: invalid syntax (344037651.py, line 1)

In [5]:
%pip install transformers==4.51.3 trl==0.15.2 peft==0.15.2 accelerate==1.2.1 datasets==3.6.0 dill==0.3.8 multiprocess==0.70.16 bitsandbytes==0.48.1 scikit-learn pandas pyarrow sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("CUDA is not available. Do not start training yet.")

Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050
VRAM: 7.93 GB


In [2]:
import transformers
import peft
import datasets
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

ImportError: huggingface-hub>=0.30.0,<1.0 is required for a normal functioning of this module, but found huggingface-hub==2.2.0.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [3]:
import traceback

try:
    import transformers
    print("Transformers:", transformers.__version__)
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_21464\635019698.py", line 4, in <module>
    import transformers
  File "C:\Users\Admin\anaconda3\Lib\site-packages\transformers\__init__.py", line 26, in <module>
    from . import dependency_versions_check
  File "C:\Users\Admin\anaconda3\Lib\site-packages\transformers\dependency_versions_check.py", line 57, in <module>
    require_version_core(deps[pkg])
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
  File "C:\Users\Admin\anaconda3\Lib\site-packages\transformers\utils\versions.py", line 117, in require_version_core
    return require_version(requirement, hint)
  File "C:\Users\Admin\anaconda3\Lib\site-packages\transformers\utils\versions.py", line 111, in require_version
    _compare_versions(op, got_ver, want_ver, requirement, pkg, hint)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\anaconda3\Lib\site-packages\transformers\utils\versions.py", line 44, in _compa

In [4]:
import sys
import subprocess

# Install a compatible Hugging Face Hub version
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall", "--no-deps",
    "huggingface-hub==0.36.2"
])

print("Hugging Face Hub installation completed.")
print("Restart the Jupyter kernel before continuing.")

Hugging Face Hub installation completed.
Restart the Jupyter kernel before continuing.


In [2]:

import os
import sys
import json
import math
import random
import subprocess
from pathlib import Path

import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Install only if required packages are missing.
required_packages = [
    "transformers",
    "peft",
    "datasets",
    "accelerate",
    "bitsandbytes",
    "scikit-learn",
    "pyarrow",
]

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is unavailable. Stop before training.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

# Keep every new experiment artifact in this separate project folder.
PROJECT_DIR = Path(r"D:\personal\research\thesis\MODEL\50%localresum")
BASE_MODEL = PROJECT_DIR / "01_base_model" / "Qwen2.5-3B-Instruct"
DATA_DIR = PROJECT_DIR / "03_dataset"
OUTPUT_DIR = PROJECT_DIR / "05_training" / "fresh_50_percent"
FINAL_DIR = PROJECT_DIR / "06_final_model" / "fresh_50_percent"

for folder in [DATA_DIR, OUTPUT_DIR, FINAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Install missing Python packages in the active Jupyter kernel.
try:
    import transformers, peft, datasets, accelerate, bitsandbytes, sklearn, pyarrow
except ImportError as exc:
    raise RuntimeError(
        f"Missing package: {exc}. Install it in the active kernel, "
        "restart the kernel, and rerun this cell."
    ) from exc

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)

# Use the previously downloaded base model, without downloading it again.
required_model_files = [
    BASE_MODEL / "config.json",
    BASE_MODEL / "model.safetensors.index.json",
]
if not all(p.exists() for p in required_model_files):
    raise FileNotFoundError(
        f"Base model is incomplete or path is wrong: {BASE_MODEL}\n"
        "Update BASE_MODEL to the actual existing model folder."
    )

# Dataset is already available locally as Parquet files.
TRAIN_FILE = DATA_DIR / "train.parquet"
VALIDATION_FILE = DATA_DIR / "validation.parquet"
TEST_FILE = DATA_DIR / "test.parquet"

if not all(p.exists() for p in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE]):
    raise FileNotFoundError(
        "Place train.parquet, validation.parquet, and test.parquet in:\n"
        f"{DATA_DIR}\n"
        "Use the Parquet files you downloaded earlier from the dataset repository."
    )

train_df = pd.read_parquet(TRAIN_FILE)
val_df = pd.read_parquet(VALIDATION_FILE)
test_df = pd.read_parquet(TEST_FILE)

assert {"text", "label"}.issubset(train_df.columns)
assert {"text", "label"}.issubset(val_df.columns)
assert {"text", "label"}.issubset(test_df.columns)

# Use exactly 50% of the training examples with balanced class sampling.
SEED = 42
rng = np.random.default_rng(SEED)
train_df["label"] = train_df["label"].astype(int)
val_df["label"] = val_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

class_ids = sorted(train_df["label"].unique().tolist())
num_classes = len(class_ids)

parts = []
for label_id in class_ids:
    group = train_df[train_df["label"] == label_id]
    n = max(1, int(len(group) * 0.50))
    selected = rng.choice(group.index.to_numpy(), size=n, replace=False)
    parts.append(train_df.loc[selected])

train_50 = (
    pd.concat(parts)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

print("\nClasses:", class_ids)
print("Class count:", num_classes)
print("Original train rows:", len(train_df))
print("50% train rows:", len(train_50))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))
print("\nTraining class distribution:")
print(train_50["label"].value_counts().sort_index())

train_50.to_parquet(DATA_DIR / "train_50_percent.parquet", index=False)

# Convert to Hugging Face datasets.
train_ds = Dataset.from_pandas(train_50, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)


Python: C:\Users\Admin\anaconda3\python.exe
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050
VRAM: 7.93 GB
Transformers: 4.51.3
PEFT: 0.15.2
Datasets: 3.6.0


FileNotFoundError: Base model is incomplete or path is wrong: D:\personal\research\thesis\MODEL\50%localresum\01_base_model\Qwen2.5-3B-Instruct
Update BASE_MODEL to the actual existing model folder.

In [1]:
import peft
import datasets
import accelerate
import bitsandbytes

print("All required ML packages imported successfully.")
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

W1012 05:45:45.581000 4544 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


All required ML packages imported successfully.
PEFT: 0.15.2
Datasets: 3.6.0
Accelerate: 1.2.1
BitsAndBytes: 0.48.1


In [3]:

from pathlib import Path

SEARCH_ROOTS = [
    Path(r"D:\personal\research\thesis\MODEL"),
    Path(r"D:\LLM\Models"),
]

print("Searching for existing model files...\n")

found_configs = []

for root in SEARCH_ROOTS:
    if not root.exists():
        print("Folder not found:", root)
        continue

    print("Searching:", root)

    for config_file in root.rglob("config.json"):
        model_dir = config_file.parent

        has_weights = (
            any(model_dir.glob("model*.safetensors"))
            or any(model_dir.glob("pytorch_model*.bin"))
            or any(model_dir.glob("model*.safetensors.index.json"))
        )

        if has_weights:
            found_configs.append(model_dir)
            print("\nModel folder:", model_dir)
            print("Config exists:", config_file.exists())
            print(
                "Weight files:",
                [p.name for p in model_dir.glob("*.safetensors")][:5]
            )

if not found_configs:
    print("\nNo complete model folder found in the searched locations.")
else:
    print("\nTotal candidate folders:", len(found_configs))


Searching for existing model files...

Searching: D:\personal\research\thesis\MODEL
Searching: D:\LLM\Models

Model folder: D:\LLM\Models\Qwen2.5-3B-Instruct
Config exists: True
Weight files: ['model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors']

Total candidate folders: 1


In [4]:

from pathlib import Path

BASE_MODEL = Path(r"D:\LLM\Models\Qwen2.5-3B-Instruct")

assert BASE_MODEL.is_dir(), f"Model folder not found: {BASE_MODEL}"
assert (BASE_MODEL / "config.json").is_file(), "config.json missing"

weight_files = list(BASE_MODEL.glob("*.safetensors"))
assert weight_files, "No model weight files found"

print("Base model verified!")
print("Path:", BASE_MODEL)
print("Weight files:")
for file in weight_files:
    print(f"  {file.name}: {file.stat().st_size / (1024**3):.2f} GB")


Base model verified!
Path: D:\LLM\Models\Qwen2.5-3B-Instruct
Weight files:
  model-00001-of-00002.safetensors: 3.70 GB
  model-00002-of-00002.safetensors: 2.05 GB


In [5]:

from pathlib import Path

ROOT = Path(r"D:\personal\research\thesis\MODEL\50%localresum")

DIRS = {
    "base_model": ROOT / "01_base_model" / "Qwen2.5-3B-Instruct",
    "hf_cache": ROOT / "02_hf_cache",
    "dataset": ROOT / "03_dataset",
    "checkpoints": ROOT / "05_training" / "checkpoints",
    "final_adapter": ROOT / "06_final_adapter",
    "results": ROOT / "07_results",
}

for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"{name}: {path}")

print("\nFolder setup complete.")
print("Existing source model remains unchanged:", BASE_MODEL)


base_model: D:\personal\research\thesis\MODEL\50%localresum\01_base_model\Qwen2.5-3B-Instruct
hf_cache: D:\personal\research\thesis\MODEL\50%localresum\02_hf_cache
dataset: D:\personal\research\thesis\MODEL\50%localresum\03_dataset
checkpoints: D:\personal\research\thesis\MODEL\50%localresum\05_training\checkpoints
final_adapter: D:\personal\research\thesis\MODEL\50%localresum\06_final_adapter
results: D:\personal\research\thesis\MODEL\50%localresum\07_results

Folder setup complete.
Existing source model remains unchanged: D:\LLM\Models\Qwen2.5-3B-Instruct


In [6]:

import sys
import subprocess
import importlib.util

# Verify that the notebook uses the intended environment
print("Python executable:", sys.executable)

required = [
    "torch", "transformers", "peft", "datasets",
    "accelerate", "bitsandbytes", "pandas", "sklearn"
]
missing = [p for p in required if importlib.util.find_spec(p) is None]

if missing:
    raise ImportError(
        f"Missing packages: {missing}\n"
        "Install them from the Anaconda Prompt using this environment's "
        "python.exe, then restart the kernel."
    )

import torch
import transformers
import peft
import datasets
import accelerate
import bitsandbytes
import pandas as pd
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Stop here; do not start training on CPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


Python executable: C:\Users\Admin\anaconda3\python.exe
PyTorch: 2.11.0+cu128
Transformers: 4.51.3
PEFT: 0.15.2
Datasets: 3.6.0
Accelerate: 1.2.1
BitsAndBytes: 0.48.1
CUDA available: True
GPU: NVIDIA GeForce RTX 5050
VRAM: 7.93 GB


In [7]:

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

ROOT = Path(r"D:\personal\research\thesis\MODEL\50%localresum")
DATA_DIR = ROOT / "03_dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Prefer existing downloaded files; otherwise download from the dataset repository
files = {}
for split in ["train", "validation", "test"]:
    local_file = DATA_DIR / f"{split}.parquet"
    if not local_file.exists():
        try:
            from huggingface_hub import hf_hub_download
            downloaded = hf_hub_download(
                repo_id="nahid-hub/BLUGE-bengali-news-classification",
                filename=f"{split}.parquet",
                repo_type="dataset",
                cache_dir=str(ROOT / "02_hf_cache")
            )
            local_file = Path(downloaded)
        except Exception as e:
            raise RuntimeError(
                f"Could not find/download {split}.parquet. "
                f"Check internet/repository access. Details: {e}"
            ) from e
    files[split] = pd.read_parquet(local_file)

train_df = files["train"].dropna(subset=["text", "label"]).copy()
val_df = files["validation"].dropna(subset=["text", "label"]).copy()
test_df = files["test"].dropna(subset=["text", "label"]).copy()

for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)

    # Remove empty text rows
    df.drop(df[df["text"].eq("")].index, inplace=True)

classes = sorted(train_df["label"].unique().tolist())
assert len(classes) == 7, f"Expected 7 classes, found {classes}"
assert sorted(val_df["label"].unique().tolist()) == classes
assert sorted(test_df["label"].unique().tolist()) == classes

# Select 50% of the training split, preserving class balance
train_50, _ = train_test_split(
    train_df,
    train_size=0.50,
    random_state=42,
    stratify=train_df["label"]
)

train_50 = train_50.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_50.to_parquet(DATA_DIR / "train_50_percent.parquet", index=False)
val_df.to_parquet(DATA_DIR / "validation.parquet", index=False)
test_df.to_parquet(DATA_DIR / "test.parquet", index=False)

print("Classes:", classes)
print("50% train:", len(train_50))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("\nTraining label counts:")
print(train_50["label"].value_counts().sort_index())
print("\nSaved dataset files to:", DATA_DIR)


train.parquet:   0%|          | 0.00/80.3M [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/10.0M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

Classes: [0, 1, 2, 3, 4, 5, 6]
50% train: 24920
Validation: 6230
Test: 6230

Training label counts:
label
0    3560
1    3560
2    3560
3    3560
4    3560
5    3560
6    3560
Name: count, dtype: int64

Saved dataset files to: D:\personal\research\thesis\MODEL\50%localresum\03_dataset


In [8]:

import json
import inspect
import numpy as np
import torch

from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

ROOT = Path(r"D:\personal\research\thesis\MODEL\50%localresum")
BASE_MODEL = Path(r"D:\LLM\Models\Qwen2.5-3B-Instruct")
CHECKPOINT_DIR = ROOT / "05_training" / "checkpoints"
FINAL_DIR = ROOT / "06_final_adapter"
RESULT_DIR = ROOT / "07_results"

for p in [CHECKPOINT_DIR, FINAL_DIR, RESULT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert (BASE_MODEL / "config.json").is_file()
assert len(classes) == 7

# Classification task: predict one of seven news labels
tokenizer = AutoTokenizer.from_pretrained(
    str(BASE_MODEL), local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForSequenceClassification.from_pretrained(
    str(BASE_MODEL),
    num_labels=len(classes),
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
    local_files_only=True,
    ignore_mismatched_sizes=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=False
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    modules_to_save=["score"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Use the existing label IDs (0..6)
def tokenize_batch(batch):
    result = tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )
    result["labels"] = [int(x) for x in batch["label"]]
    return result

train_ds = Dataset.from_pandas(train_50[["text", "label"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text", "label"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text", "label"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["text", "label"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

args_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    eval_strategy="epoch",
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing=False,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    optim="paged_adamw_8bit",
    seed=42,
)

# Compatibility with Transformers versions that use evaluation_strategy
if "eval_strategy" not in inspect.signature(TrainingArguments.__init__).parameters:
    args_kwargs["evaluation_strategy"] = args_kwargs.pop("eval_strategy")

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Train from scratch; do not resume from an old checkpoint
train_result = trainer.train(resume_from_checkpoint=False)

# Save adapter and tokenizer separately from the base model
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

# Final evaluation on held-out test split
test_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
with open(RESULT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2, ensure_ascii=False)

with open(RESULT_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, indent=2, ensure_ascii=False)

print("Training completed.")
print("Final adapter:", FINAL_DIR)
print("Test metrics:", test_metrics)
print("Metrics saved to:", RESULT_DIR)


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

C:\Users\Admin\anaconda3\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at D:\LLM\Models\Qwen2.5-3B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,857,536 || all params: 3,087,810,560 || trainable%: 0.0602


TypeError: Pickler._batch_setitems() takes 2 positional arguments but 3 were given

In [9]:

from datasets import Dataset
import pandas as pd

# Avoid pandas/Arrow table fingerprint serialization.
# Build datasets from plain Python lists instead.
def dataframe_to_dataset(df):
    return Dataset.from_dict({
        "text": df["text"].astype(str).tolist(),
        "label": df["label"].astype(int).tolist(),
    })

train_ds = dataframe_to_dataset(train_50)
val_ds = dataframe_to_dataset(val_df)
test_ds = dataframe_to_dataset(test_df)

def tokenize_batch(batch):
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )
    encoded["labels"] = [int(x) for x in batch["label"]]
    return encoded

train_ds = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label"],
    load_from_cache_file=False,
    new_fingerprint="train50_tokenized_v1",
)

val_ds = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label"],
    load_from_cache_file=False,
    new_fingerprint="validation_tokenized_v1",
)

test_ds = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label"],
    load_from_cache_file=False,
    new_fingerprint="test_tokenized_v1",
)

print("Tokenization successful.")
print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))
print("Sample keys:", train_ds[0].keys())
print("Sample label:", train_ds[0]["labels"])


TypeError: Pickler._batch_setitems() takes 2 positional arguments but 3 were given

In [10]:

import pandas as pd
from datasets import Dataset

# Validate the input data
required_columns = {"text", "label"}
for name, df in [
    ("train_50", train_50),
    ("val_df", val_df),
    ("test_df", test_df),
]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

# Convert through plain Python lists to avoid the pandas Arrow
# table serialization path that triggered your previous error.
def dataframe_to_dataset(df):
    clean = df[["text", "label"]].dropna().copy()
    clean["text"] = clean["text"].astype(str).str.strip()
    clean["label"] = clean["label"].astype(int)
    clean = clean[clean["text"] != ""]
    return Dataset.from_dict({
        "text": clean["text"].tolist(),
        "label": clean["label"].tolist(),
    })

train_ds = dataframe_to_dataset(train_50)
val_ds = dataframe_to_dataset(val_df)
test_ds = dataframe_to_dataset(test_df)

# Confirm all splits have the same 7 label IDs
expected_labels = list(range(7))
for name, ds in [
    ("train", train_ds),
    ("validation", val_ds),
    ("test", test_ds),
]:
    found = sorted(set(ds["label"]))
    print(f"{name}: {len(ds)} rows; labels={found}")
    if found != expected_labels:
        raise ValueError(f"Unexpected labels in {name}: {found}")

def tokenize_batch(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )
    enc["labels"] = [int(x) for x in batch["label"]]
    return enc

train_ds = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label"],
    load_from_cache_file=False,
)
val_ds = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label"],
    load_from_cache_file=False,
)
test_ds = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label"],
    load_from_cache_file=False,
)

print("\nTokenization completed.")
print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))
print("Sample token length:", len(train_ds[0]["input_ids"]))
print("Sample label:", train_ds[0]["labels"])


TypeError: Pickler._batch_setitems() takes 2 positional arguments but 3 were given

In [11]:

import torch
import pandas as pd
from torch.utils.data import Dataset as TorchDataset

MAX_LENGTH = 256
NUM_CLASSES = 7

# Verify that all required objects are available
required_objects = [
    "train_50", "val_df", "test_df", "tokenizer", "model"
]

for name in required_objects:
    if name not in globals():
        raise NameError(
            f"Missing object: {name}. Run the relevant setup cell first."
        )

# Clean the DataFrame without using Hugging Face Dataset
def clean_dataframe(df):
    cleaned = df[["text", "label"]].dropna().copy()
    cleaned["text"] = cleaned["text"].astype(str).str.strip()
    cleaned["label"] = cleaned["label"].astype(int)
    cleaned = cleaned[cleaned["text"] != ""].reset_index(drop=True)

    if not cleaned["label"].isin(range(NUM_CLASSES)).all():
        raise ValueError("Labels must be integers from 0 to 6.")

    return cleaned

train_clean = clean_dataframe(train_50)
val_clean = clean_dataframe(val_df)
test_clean = clean_dataframe(test_df)

# Create a native PyTorch dataset
class NewsClassificationDataset(TorchDataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoded.items()
        }
        item["labels"] = torch.tensor(
            self.labels[index], dtype=torch.long
        )
        return item

# Build training, validation, and test datasets
train_ds = NewsClassificationDataset(
    train_clean, tokenizer, MAX_LENGTH
)
val_ds = NewsClassificationDataset(
    val_clean, tokenizer, MAX_LENGTH
)
test_ds = NewsClassificationDataset(
    test_clean, tokenizer, MAX_LENGTH
)

# Verify dataset sizes and label distribution
print("Training examples:", len(train_ds))
print("Validation examples:", len(val_ds))
print("Test examples:", len(test_ds))
print("Number of classes:", len(set(train_clean["label"])))
print("Class IDs:", sorted(train_clean["label"].unique().tolist()))
print("Sample input shape:", train_ds[0]["input_ids"].shape)
print("Sample label:", train_ds[0]["labels"].item())

assert len(train_ds) > 0
assert len(val_ds) > 0
assert len(test_ds) > 0
assert set(train_clean["label"].unique()) == set(range(NUM_CLASSES))

print("Dataset verification passed.")


Training examples: 24920
Validation examples: 6230
Test examples: 6230
Number of classes: 7
Class IDs: [0, 1, 2, 3, 4, 5, 6]
Sample input shape: torch.Size([256])
Sample label: 2
Dataset verification passed.


In [12]:

import json
import inspect
import numpy as np
import torch

from pathlib import Path
from transformers import TrainingArguments, Trainer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# Define project directories
ROOT = Path(r"D:\personal\research\thesis\MODEL\50%localresum")

CHECKPOINT_DIR = ROOT / "05_training" / "checkpoints"
FINAL_DIR = ROOT / "06_final_adapter"
RESULT_DIR = ROOT / "07_results"

for directory in [CHECKPOINT_DIR, FINAL_DIR, RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Verify GPU availability
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Check the active Jupyter kernel and PyTorch installation."
    )

# Verify that the model contains a PEFT adapter
if not hasattr(model, "peft_config"):
    raise TypeError(
        "The model has no PEFT adapter. Verify the QLoRA model setup."
    )

# Verify that this is a sequence-classification model
if getattr(model.config, "num_labels", None) != NUM_CLASSES:
    raise ValueError(
        f"Expected {NUM_CLASSES} output classes, "
        f"but model.config.num_labels={model.config.num_labels}."
    )

# Disable gradient checkpointing to avoid the previously observed
# checkpoint recomputation error during backpropagation
model.config.use_cache = False
model.gradient_checkpointing_disable()

use_bf16 = torch.cuda.is_bf16_supported()

# Calculate accuracy, macro precision, recall, and F1
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
    }

# Configure training parameters
training_config = {
    "output_dir": str(CHECKPOINT_DIR),
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "logging_steps": 25,
    "save_strategy": "steps",
    "save_steps": 500,
    "save_total_limit": 3,
    "fp16": not use_bf16,
    "bf16": use_bf16,
    "gradient_checkpointing": False,
    "report_to": "none",
    "dataloader_num_workers": 0,
    "remove_unused_columns": False,
    "optim": "paged_adamw_8bit",
    "seed": 42,
}

# Support both older and newer Transformers argument names
available_args = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in available_args:
    training_config["eval_strategy"] = "epoch"
else:
    training_config["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_config)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

expected_steps = (
    len(train_ds)
    + training_args.gradient_accumulation_steps - 1
) // training_args.gradient_accumulation_steps

print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(train_ds))
print("Expected optimizer steps:", expected_steps)
print("Gradient checkpointing:", model.is_gradient_checkpointing)
print("Output directory:", CHECKPOINT_DIR)

# Start a fresh training run; do not resume an old checkpoint
train_result = trainer.train(resume_from_checkpoint=False)

# Save the final PEFT adapter and tokenizer
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

# Evaluate only on the held-out test set
test_result = trainer.predict(
    test_ds,
    metric_key_prefix="test",
)

test_predictions = np.argmax(
    test_result.predictions, axis=-1
)
test_labels = test_result.label_ids

# Generate detailed classification metrics
report = classification_report(
    test_labels,
    test_predictions,
    labels=list(range(NUM_CLASSES)),
    output_dict=True,
    zero_division=0,
)

matrix = confusion_matrix(
    test_labels,
    test_predictions,
    labels=list(range(NUM_CLASSES)),
)

results = {
    "train_metrics": train_result.metrics,
    "test_metrics": test_result.metrics,
    "classification_report": report,
    "confusion_matrix": matrix.tolist(),
    "training_examples": len(train_ds),
    "validation_examples": len(val_ds),
    "test_examples": len(test_ds),
    "num_classes": NUM_CLASSES,
    "max_sequence_length": MAX_LENGTH,
}

# Write all evaluation results to a JSON file
with open(
    RESULT_DIR / "all_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        results,
        file,
        indent=2,
        ensure_ascii=False,
        default=float,
    )

print("\nTraining completed.")
print("Final adapter:", FINAL_DIR)
print("Evaluation results:", RESULT_DIR / "all_metrics.json")
print("\nTest metrics:")
print(test_result.metrics)

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        labels=list(range(NUM_CLASSES)),
        zero_division=0,
    )
)


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


GPU: NVIDIA GeForce RTX 5050
Training examples: 24920
Expected optimizer steps: 3115
Gradient checkpointing: False
Output directory: D:\personal\research\thesis\MODEL\50%localresum\05_training\checkpoints


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.382700,0.394135,0.883628,0.883225,0.883628,0.883349



Training completed.
Final adapter: D:\personal\research\thesis\MODEL\50%localresum\06_final_adapter
Evaluation results: D:\personal\research\thesis\MODEL\50%localresum\07_results\all_metrics.json

Test metrics:
{'test_loss': 0.4233109951019287, 'test_accuracy': 0.8764044943820225, 'test_macro_precision': 0.8767383592240471, 'test_macro_recall': 0.8764044943820225, 'test_macro_f1': 0.876441021974156, 'test_runtime': 886.1272, 'test_samples_per_second': 7.031, 'test_steps_per_second': 7.031}

Classification report:
              precision    recall  f1-score   support

           0       0.90      0.89      0.89       890
           1       0.83      0.82      0.82       890
           2       0.96      0.95      0.96       890
           3       0.86      0.90      0.88       890
           4       0.89      0.86      0.87       890
           5       0.93      0.95      0.94       890
           6       0.77      0.78      0.78       890

    accuracy                           0.88   

In [13]:

# ============================================================
# Comprehensive Evaluation for Bengali News Classification
# Metrics: Macro-F1, Macro Precision, Macro Recall,
# Balanced Accuracy, Accuracy, MCC, Weighted-F1, Confusion Matrix
# ============================================================

import json
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path(
    r"D:\personal\research\thesis\MODEL\50%localresum"
)

RESULT_DIR = ROOT / "07_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 7
CLASS_LABELS = list(range(NUM_CLASSES))

# Use the trained model already loaded in the notebook.
model.eval()
device = next(model.parameters()).device

# ------------------------------------------------------------
# 2. Prepare test inputs and true labels
# ------------------------------------------------------------

# test_ds should contain "text" and "label" columns.
test_texts = list(test_ds["text"])
y_true = np.asarray(test_ds["label"], dtype=np.int64)

# ------------------------------------------------------------
# 3. Generate predictions in batches
# ------------------------------------------------------------

BATCH_SIZE = 4
MAX_LENGTH = 256
all_predictions = []

for start in range(0, len(test_texts), BATCH_SIZE):
    batch_texts = test_texts[start:start + BATCH_SIZE]

    # Tokenize the current batch.
    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    # Disable gradients for inference.
    with torch.inference_mode():
        outputs = model(**encoded)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

    all_predictions.extend(
        predictions.cpu().numpy().tolist()
    )

y_pred = np.asarray(all_predictions, dtype=np.int64)

assert len(y_true) == len(y_pred), (
    f"Length mismatch: true={len(y_true)}, predicted={len(y_pred)}"
)

# ------------------------------------------------------------
# 4. Calculate evaluation metrics
# ------------------------------------------------------------

metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_precision": float(
        precision_score(
            y_true, y_pred,
            labels=CLASS_LABELS,
            average="macro",
            zero_division=0,
        )
    ),
    "macro_recall": float(
        recall_score(
            y_true, y_pred,
            labels=CLASS_LABELS,
            average="macro",
            zero_division=0,
        )
    ),
    "macro_f1": float(
        f1_score(
            y_true, y_pred,
            labels=CLASS_LABELS,
            average="macro",
            zero_division=0,
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_true, y_pred)
    ),
    "mcc": float(
        matthews_corrcoef(y_true, y_pred)
    ),
    "weighted_f1": float(
        f1_score(
            y_true, y_pred,
            labels=CLASS_LABELS,
            average="weighted",
            zero_division=0,
        )
    ),
}

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASS_LABELS,
)

report = classification_report(
    y_true,
    y_pred,
    labels=CLASS_LABELS,
    output_dict=True,
    zero_division=0,
)

# ------------------------------------------------------------
# 5. Display metrics
# ------------------------------------------------------------

print("=" * 55)
print("BENGALI NEWS CLASSIFICATION — TEST RESULTS")
print("=" * 55)

for name, value in metrics.items():
    print(f"{name:22s}: {value:.6f} ({value * 100:.2f}%)")

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix:")
print(cm)

# ------------------------------------------------------------
# 6. Plot confusion matrix
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 7))

image = ax.imshow(cm, interpolation="nearest")
fig.colorbar(image, ax=ax)

ax.set(
    title="Confusion Matrix — Bengali News Classification",
    xlabel="Predicted Label",
    ylabel="True Label",
    xticks=CLASS_LABELS,
    yticks=CLASS_LABELS,
)

threshold = cm.max() / 2 if cm.size else 0

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center",
            va="center",
            color="white" if cm[i, j] > threshold else "black",
        )

fig.tight_layout()

cm_path = RESULT_DIR / "confusion_matrix.png"
fig.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 7. Save metrics, report, and predictions
# ------------------------------------------------------------

results = {
    "evaluation_split": "test",
    "number_of_test_examples": int(len(y_true)),
    "number_of_classes": NUM_CLASSES,
    "metrics": metrics,
    "classification_report": report,
    "confusion_matrix": cm.tolist(),
}

with open(
    RESULT_DIR / "comprehensive_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        results,
        file,
        indent=2,
        ensure_ascii=False,
    )

np.savez_compressed(
    RESULT_DIR / "test_predictions.npz",
    y_true=y_true,
    y_pred=y_pred,
)

print("\nEvaluation completed.")
print("Metrics:", RESULT_DIR / "comprehensive_metrics.json")
print("Confusion matrix:", cm_path)
print("Predictions:", RESULT_DIR / "test_predictions.npz")


TypeError: list indices must be integers or slices, not str

In [14]:

# ============================================================
# Evaluation Metrics for the Existing Custom Test Dataset
# ============================================================

import json
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
)

# 1. Configure output directory
ROOT = Path(
    r"D:\personal\research\thesis\MODEL\50%localresum"
)
RESULT_DIR = ROOT / "07_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 7
CLASS_LABELS = list(range(NUM_CLASSES))
BATCH_SIZE = 8

# 2. Confirm the custom dataset and model
print("Test dataset size:", len(test_ds))
print("Dataset class:", type(test_ds).__name__)

model.eval()
device = next(model.parameters()).device

# 3. Run inference using the custom dataset
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

y_true = []
y_pred = []

with torch.inference_mode():
    for batch in test_loader:
        # Custom Dataset returns encoded inputs and labels.
        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model(**inputs)
        predictions = outputs.logits.argmax(dim=-1)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(predictions.cpu().numpy().tolist())

y_true = np.asarray(y_true, dtype=np.int64)
y_pred = np.asarray(y_pred, dtype=np.int64)

assert len(y_true) == len(y_pred)
assert len(y_true) > 0

# 4. Calculate metrics
metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_precision": float(
        precision_score(
            y_true, y_pred, labels=CLASS_LABELS,
            average="macro", zero_division=0
        )
    ),
    "macro_recall": float(
        recall_score(
            y_true, y_pred, labels=CLASS_LABELS,
            average="macro", zero_division=0
        )
    ),
    "macro_f1": float(
        f1_score(
            y_true, y_pred, labels=CLASS_LABELS,
            average="macro", zero_division=0
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_true, y_pred)
    ),
    "mcc": float(matthews_corrcoef(y_true, y_pred)),
    "weighted_f1": float(
        f1_score(
            y_true, y_pred, labels=CLASS_LABELS,
            average="weighted", zero_division=0
        )
    ),
}

cm = confusion_matrix(
    y_true, y_pred, labels=CLASS_LABELS
)

# 5. Display results
print("\n========== TEST METRICS ==========")
for name, value in metrics.items():
    print(f"{name:22s}: {value:.6f}")

print("\n========== CLASSIFICATION REPORT ==========")
report_text = classification_report(
    y_true,
    y_pred,
    labels=CLASS_LABELS,
    digits=4,
    zero_division=0,
)
print(report_text)

print("\n========== CONFUSION MATRIX ==========")
print(cm)

# 6. Plot and save confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(cm, interpolation="nearest")
fig.colorbar(image, ax=ax)

ax.set(
    title="Bengali News Classification - Confusion Matrix",
    xlabel="Predicted Label",
    ylabel="True Label",
    xticks=CLASS_LABELS,
    yticks=CLASS_LABELS,
)

threshold = cm.max() / 2
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center",
            va="center",
            color="white" if cm[i, j] > threshold else "black",
        )

fig.tight_layout()
fig.savefig(
    RESULT_DIR / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# 7. Save metrics and predictions
results = {
    "number_of_test_examples": int(len(y_true)),
    "number_of_classes": NUM_CLASSES,
    "metrics": metrics,
    "confusion_matrix": cm.tolist(),
    "classification_report": classification_report(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        output_dict=True,
        zero_division=0,
    ),
}

with open(
    RESULT_DIR / "comprehensive_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(results, file, indent=2, ensure_ascii=False)

np.savez_compressed(
    RESULT_DIR / "test_predictions.npz",
    
    y_true=y_true,
    y_pred=y_pred,
)

print("\nEvaluation complete.")
print("Saved results to:", RESULT_DIR)


Test dataset size: 6230
Dataset class: NewsClassificationDataset


KeyboardInterrupt: 

In [15]:

import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

# --------------------------------------------------
# 1. Configure output directory
# --------------------------------------------------
RESULT_DIR = Path(DIRS["results"])
RESULT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Test examples:", len(test_ds))
print("Results directory:", RESULT_DIR)

# --------------------------------------------------
# 2. Prepare the test DataLoader
# --------------------------------------------------
test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# --------------------------------------------------
# 3. Run model inference
# --------------------------------------------------
model.eval()

all_predictions = []
all_labels = []

with torch.inference_mode():
    for step, batch in enumerate(test_loader, start=1):

        # Support the existing custom Dataset class
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        all_predictions.extend(predictions.cpu().numpy().tolist())
        all_labels.extend(
            labels.cpu().numpy().astype(int).tolist()
        )

        if step % 50 == 0 or step == len(test_loader):
            print(
                f"Progress: {step}/{len(test_loader)} batches"
            )

y_true = np.asarray(all_labels, dtype=np.int64)
y_pred = np.asarray(all_predictions, dtype=np.int64)

assert len(y_true) == len(test_ds), "Test label count mismatch!"
assert len(y_pred) == len(y_true), "Prediction count mismatch!"

# --------------------------------------------------
# 4. Calculate evaluation metrics
# --------------------------------------------------
metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_precision": float(
        precision_score(y_true, y_pred, average="macro", zero_division=0)
    ),
    "macro_recall": float(
        recall_score(y_true, y_pred, average="macro", zero_division=0)
    ),
    "macro_f1": float(
        f1_score(y_true, y_pred, average="macro", zero_division=0)
    ),
    "weighted_f1": float(
        f1_score(y_true, y_pred, average="weighted", zero_division=0)
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_true, y_pred)
    ),
    "mcc": float(matthews_corrcoef(y_true, y_pred)),
}

cm = confusion_matrix(y_true, y_pred)
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    output_dict=True,
    zero_division=0
)

# --------------------------------------------------
# 5. Save metrics and confusion matrix
# --------------------------------------------------
with open(
    RESULT_DIR / "comprehensive_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {"metrics": metrics, "classification_report": report},
        f,
        indent=2,
        ensure_ascii=False
    )

np.savez_compressed(
    RESULT_DIR / "test_predictions.npz",
    y_true=y_true,
    y_pred=y_pred
)

np.savetxt(
    RESULT_DIR / "confusion_matrix.csv",
    cm,
    delimiter=",",
    fmt="%d"
)

# --------------------------------------------------
# 6. Plot and save the confusion matrix
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 7))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(range(NUM_CLASSES))
)
display.plot(ax=ax, cmap="Blues", values_format="d")

ax.set_title("Bangla News Classification - Confusion Matrix")
fig.tight_layout()

fig.savefig(
    RESULT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# --------------------------------------------------
# 7. Display all results
# --------------------------------------------------
print("\n========== FINAL TEST METRICS ==========")

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\n========== CLASSIFICATION REPORT ==========")
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        zero_division=0
    )
)

print("\nAll evaluation files saved to:")
print(RESULT_DIR)


Device: cuda
Test examples: 6230
Results directory: D:\personal\research\thesis\MODEL\50%localresum\07_results
Progress: 50/779 batches


KeyboardInterrupt: 

In [17]:

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef
)

# ============================================================
# 1. Configure paths
# ============================================================
RESULT_DIR = Path(
    r"D:\personal\research\thesis\MODEL\50%localresum\07_results"
)
GRAPH_DIR = RESULT_DIR / "publication_graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

# Use existing predictions if they have already been saved.
prediction_file = RESULT_DIR / "test_predictions.npz"
metrics_file = RESULT_DIR / "comprehensive_metrics.json"

if prediction_file.exists():
    saved = np.load(prediction_file)
    y_true = saved["y_true"].astype(int)
    y_pred = saved["y_pred"].astype(int)
elif "y_true" in globals() and "y_pred" in globals():
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
else:
    raise FileNotFoundError(
        "Test predictions not found. Run the evaluation cell first."
    )

assert len(y_true) == len(y_pred), "Prediction and label lengths differ!"

# Recalculate metrics from the actual predictions.
metric_values = {
    "Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "Macro-Precision": precision_score(
        y_true, y_pred, average="macro", zero_division=0
    ),
    "Macro-Recall": recall_score(
        y_true, y_pred, average="macro", zero_division=0
    ),
    "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
    "Accuracy": accuracy_score(y_true, y_pred),
    "MCC": matthews_corrcoef(y_true, y_pred),
    "Weighted-F1": f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    ),
}

labels = sorted(np.unique(np.concatenate([y_true, y_pred])).tolist())
cm = confusion_matrix(y_true, y_pred, labels=labels)

report = classification_report(
    y_true, y_pred,
    labels=labels,
    output_dict=True,
    zero_division=0
)

print("Test samples:", len(y_true))
print("Graphs will be saved to:", GRAPH_DIR)

# ============================================================
# 2. Graph 1: Overall evaluation metrics
# ============================================================
names = list(metric_values.keys())
values = list(metric_values.values())

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(names[::-1], values[::-1])

ax.set_xlim(-0.05, 1.05)
ax.set_xlabel("Score")
ax.set_title("Overall Test Performance", fontweight="bold", pad=14)
ax.grid(axis="x", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for bar, value in zip(bars, values[::-1]):
    ax.text(
        min(value + 0.015, 0.96),
        bar.get_y() + bar.get_height() / 2,
        f"{value:.4f}",
        va="center",
        fontsize=10
    )

fig.tight_layout()
fig.savefig(GRAPH_DIR / "01_overall_metrics.png", dpi=300, bbox_inches="tight")
fig.savefig(GRAPH_DIR / "01_overall_metrics.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

# ============================================================
# 3. Graph 2: Per-class Precision, Recall and F1
# ============================================================
class_names = [str(label) for label in labels]
precision_values = [report[str(i)]["precision"] for i in labels]
recall_values = [report[str(i)]["recall"] for i in labels]
f1_values = [report[str(i)]["f1-score"] for i in labels]

x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - width, precision_values, width, label="Precision")
ax.bar(x, recall_values, width, label="Recall")
ax.bar(x + width, f1_values, width, label="F1-score")

ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.set_ylim(0, 1.08)
ax.set_xlabel("Class ID")
ax.set_ylabel("Score")
ax.set_title("Per-Class Classification Performance", fontweight="bold", pad=14)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(GRAPH_DIR / "02_per_class_metrics.png", dpi=300, bbox_inches="tight")
fig.savefig(GRAPH_DIR / "02_per_class_metrics.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

# ============================================================
# 4. Graph 3: Confusion matrix (raw counts)
# ============================================================
fig, ax = plt.subplots(figsize=(9, 8))
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)
display.plot(ax=ax, cmap="Blues", values_format="d", colorbar=True)

ax.set_title("Confusion Matrix — Test Set", fontweight="bold", pad=16)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")

fig.tight_layout()
fig.savefig(GRAPH_DIR / "03_confusion_matrix.png", dpi=300, bbox_inches="tight")
fig.savefig(GRAPH_DIR / "03_confusion_matrix.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

# ============================================================
# 5. Graph 4: Normalized confusion matrix
# ============================================================
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm, row_sums,
    out=np.zeros_like(cm, dtype=float),
    where=row_sums != 0
)

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(cm_normalized, interpolation="nearest", cmap="Blues",
                  vmin=0, vmax=1)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

ax.set(
    xticks=np.arange(len(class_names)),
    yticks=np.arange(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="Predicted class",
    ylabel="True class",
    title="Normalized Confusion Matrix"
)
plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

threshold = 0.5
for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        value = cm_normalized[i, j]
        ax.text(
            j, i, f"{value:.2f}",
            ha="center", va="center",
            color="white" if value > threshold else "black",
            fontsize=9
        )

fig.tight_layout()
fig.savefig(
    GRAPH_DIR / "04_normalized_confusion_matrix.png",
    dpi=300, bbox_inches="tight"
)
fig.savefig(
    GRAPH_DIR / "04_normalized_confusion_matrix.pdf",
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ============================================================
# 6. Export metrics as CSV and JSON
# ============================================================
pd.DataFrame(
    [{"Metric": key, "Score": value} for key, value in metric_values.items()]
).to_csv(GRAPH_DIR / "evaluation_metrics.csv", index=False)

pd.DataFrame(report).T.to_csv(
    GRAPH_DIR / "per_class_classification_report.csv"
)

with open(GRAPH_DIR / "evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "number_of_test_samples": int(len(y_true)),
            "metrics": {
                key: float(value)
                for key, value in metric_values.items()
            }
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n========== FINAL TEST RESULTS ==========")
for name, value in metric_values.items():
    print(f"{name:22s}: {value:.4f}")

print("\nAll graphs and metric files saved successfully:")
print(GRAPH_DIR)
print("\nFiles:")
for file in sorted(GRAPH_DIR.iterdir()):
    print(" -", file.name)


AssertionError: Prediction and label lengths differ!

In [ ]:

import numpy as np
import torch
from torch.utils.data import DataLoader
from pathlib import Path

# Use the existing test dataset and trained model.
assert "test_ds" in globals(), "test_ds is not available."
assert "model" in globals(), "Trained model is not available."

# Get true labels from the underlying dataset.
if hasattr(test_ds, "labels"):
    y_true = np.asarray(test_ds.labels, dtype=np.int64)
elif hasattr(test_ds, "texts") and hasattr(test_ds, "labels"):
    y_true = np.asarray(test_ds.labels, dtype=np.int64)
else:
    raise AttributeError(
        "Cannot find test labels. Inspect test_ds attributes first."
    )

# Create a DataLoader using the existing dataset class.
test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

# Generate predictions from the trained model.
model.eval()
all_predictions = []

with torch.inference_mode():
    for batch in test_loader:
        # Move model inputs to the model's device.
        device = next(model.parameters()).device

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key in ("input_ids", "attention_mask", "token_type_ids")
            and torch.is_tensor(value)
        }

        outputs = model(**inputs)
        predictions = outputs.logits.argmax(dim=-1)
        all_predictions.extend(predictions.cpu().numpy().tolist())

y_pred = np.asarray(all_predictions, dtype=np.int64)

print("True labels:", len(y_true))
print("Predictions:", len(y_pred))
print("Shapes:", y_true.shape, y_pred.shape)

assert len(y_true) == len(y_pred), (
    "Mismatch remains. Check test dataset labels and DataLoader."
)

print("Prediction lengths match. Ready to generate graphs.")
